In [1]:
!pip install -q transformers datasets scikit-learn

from datasets import load_dataset
from collections import Counter
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np
import json

print("Done.")

Done.


In [2]:
dataset  = load_dataset("freococo/burmese-contextual-pragmatics")
train_ds = dataset["train"]

politeness_map = {
    "neutral":"neutral", "polite":"polite", "informal":"informal",
    "professional":"professional", "blunt":"blunt", "rude":"rude",
    "very_polite":"polite", "friendly":"informal", "religious":"polite",
    "impolite":"rude", "stern":"blunt", "sarcastic":"rude",
    "polite_but_firm":"polite"
}

positive = {"Warm","Sweet","Cheerful","Soft","Sincere","Friendly","Kind",
            "Caring","Gentle","Encouraging","Supportive","Peaceful","Grateful",
            "Romantic","Joyful","Hopeful","Welcoming","Trusting","Approving",
            "Complimentary","Amused","Excited","Moved","Nostalgic","Sympathetic",
            "Pitying","Curious","Playful","Tender","Awe-struck","Awe",
            "Grateful/Humble","Care"}
formal_t = {"Professional","Serious","Respectful","Humble","Stern","Firm",
            "Determined","Authoritative","Commanding","Objective","Deep",
            "Intense","Stern/Concerned"}
negative = {"Angry","Annoyed","Hostile","Aggressive","Harsh","Hateful",
            "Disgusted","Cold","Dismissive","Sarcastic","Impolite","Rude",
            "Blunt","Crude","Envious","Dangerous","Fed up","Dry","Indifferent"}
emotional= {"Urgent","Pleading","Dramatic","Sad","Distressed","Tired",
            "Exhausted","Panicked","Shocked","Surprised","Concerned",
            "Apologetic","Appologetic","Regretful","Emotional","Teasing",
            "Casual","Dazed","Confused","Alarmed","Overwhelmed","Startled",
            "Devastated","Worried","Nervous/Sincere","Sad/Soft","Sad/Sweet",
            "Distress","Painful","Sorrowful","Weak","Uncertain","Weary",
            "Hurry","Impatient","Relieved","Fearful","Scared","Frightened",
            "Grieved","Incredulous","Bored","Lazy","Sleepy","Stressed",
            "Exasperated","Frustrated","Mischievous","Suggestive","Spooky",
            "Whispering","Indirect","Exclamatory","Lazy/Neutral",
            "Casual/Sleepy","Cheerful/Tired","Sweet/Weary","Sad/Serious",
            "Soft/Vulnerable","Natural","Considerate","Faithful","Trusting"}

def coarsen_tone(t):
    if t in positive:  return "positive"
    if t in formal_t:  return "formal"
    if t in negative:  return "negative"
    if t in emotional: return "emotional"
    return "neutral"

power_map = {
    "equal":"equal",
    "inferior_to_superior":"inferior_to_superior",
    "superior_to_inferior":"superior_to_inferior",
    "any":"any",
    "customer_to_staff":"inferior_to_superior",
    "customer_to_seller":"inferior_to_superior",
    "passenger_to_driver":"inferior_to_superior",
    "patient_to_doctor":"inferior_to_superior",
}

def apply_all_labels(example):
    example["politeness_coarse"] = politeness_map.get(example["politeness"], "neutral")
    example["tone_coarse"]       = coarsen_tone(example["tone"])
    example["power_coarse"]      = power_map.get(example["power_relation"], "equal")
    example["register_label"]    = example["register"]
    return example

train_ds = train_ds.map(apply_all_labels)

le_pol  = LabelEncoder().fit(train_ds["politeness_coarse"])
le_reg  = LabelEncoder().fit(train_ds["register_label"])
le_pow  = LabelEncoder().fit(train_ds["power_coarse"])
le_tone = LabelEncoder().fit(train_ds["tone_coarse"])

print("Politeness:", list(le_pol.classes_))
print("Register:  ", list(le_reg.classes_))
print("Power:     ", list(le_pow.classes_))
print("Tone:      ", list(le_tone.classes_))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

burmese-conversational-pragmatics.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/2200 [00:00<?, ? examples/s]

Map:   0%|          | 0/2200 [00:00<?, ? examples/s]

Politeness: [np.str_('blunt'), np.str_('informal'), np.str_('neutral'), np.str_('polite'), np.str_('professional'), np.str_('rude')]
Register:   [np.str_('colloquial'), np.str_('formal'), np.str_('slang'), np.str_('standard')]
Power:      [np.str_('any'), np.str_('equal'), np.str_('inferior_to_superior'), np.str_('superior_to_inferior')]
Tone:       [np.str_('emotional'), np.str_('formal'), np.str_('negative'), np.str_('neutral'), np.str_('positive')]


In [3]:
def encode_all_labels(example):
    example["label_pol"]  = int(le_pol.transform([example["politeness_coarse"]])[0])
    example["label_reg"]  = int(le_reg.transform([example["register_label"]])[0])
    example["label_pow"]  = int(le_pow.transform([example["power_coarse"]])[0])
    example["label_tone"] = int(le_tone.transform([example["tone_coarse"]])[0])
    return example

train_ds = train_ds.map(encode_all_labels)

split1     = train_ds.train_test_split(test_size=0.30, seed=42)
split2     = split1["test"].train_test_split(test_size=0.50, seed=42)
train_data = split1["train"]
val_data   = split2["train"]
test_data  = split2["test"]

print(f"Train: {len(train_data)}  Val: {len(val_data)}  Test: {len(test_data)}")

def get_weights(data, label_col, n_classes):
    counts = Counter(data[label_col])
    total  = len(data)
    return torch.tensor(
        [total / (n_classes * counts[i]) for i in range(n_classes)],
        dtype=torch.float
    )

w_pol  = get_weights(train_data, "label_pol",  len(le_pol.classes_))
w_reg  = get_weights(train_data, "label_reg",  len(le_reg.classes_))
w_pow  = get_weights(train_data, "label_pow",  len(le_pow.classes_))
w_tone = get_weights(train_data, "label_tone", len(le_tone.classes_))

print("Weights pol: ", [round(x,3) for x in w_pol.tolist()])
print("Weights reg: ", [round(x,3) for x in w_reg.tolist()])
print("Weights pow: ", [round(x,3) for x in w_pow.tolist()])
print("Weights tone:", [round(x,3) for x in w_tone.tolist()])

Map:   0%|          | 0/2200 [00:00<?, ? examples/s]

Train: 1540  Val: 330  Test: 330
Weights pol:  [5.238, 0.976, 0.318, 0.817, 3.889, 6.111]
Weights reg:  [0.364, 4.01, 3.565, 1.39]
Weights pow:  [3.377, 0.33, 2.567, 3.5]
Weights tone: [1.149, 1.257, 4.738, 0.487, 0.936]


In [4]:
MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 128
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device: cuda


In [5]:
# ── Load all models ───────────────────────────────────────────────────────────
print("Loading Classifier A...")
model_A = AutoModelForSequenceClassification.from_pretrained(
    "annasus10/xlmr-burmese-register").to(device).eval()

print("Loading Classifier B...")
model_B = AutoModelForSequenceClassification.from_pretrained(
    "annasus10/xlmr-burmese-power").to(device).eval()

print("Loading Classifier C...")
model_C = AutoModelForSequenceClassification.from_pretrained(
    "annasus10/xlmr-burmese-tone").to(device).eval()

print("Loading Stage 3 politeness model...")
model_pol = AutoModelForSequenceClassification.from_pretrained(
    "annasus10/xlmr-burmese-pragmatics-stage3-v2").to(device).eval()

print("All models loaded.")

Loading Classifier A...


config.json:   0%|          | 0.00/954 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading Classifier B...


...

Loading Stage 3 politeness model...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

All models loaded.


In [6]:
# ── Helpers ───────────────────────────────────────────────────────────────────

reg_classes  = list(le_reg.classes_)
pow_classes  = list(le_pow.classes_)
tone_classes = list(le_tone.classes_)
pol_classes  = list(le_pol.classes_)

def predict_single(model, text):
    enc = tokenizer(text, max_length=MAX_LENGTH, truncation=True,
                    padding="max_length", return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**enc).logits
    return torch.argmax(logits, dim=1).item()

def get_oracle_metadata(example):
    reg  = example["register"]
    pow_ = power_map.get(example["power_relation"], "equal")
    tone = coarsen_tone(example["tone"])
    return reg, pow_, tone

print("Helpers ready.")

Helpers ready.


In [7]:
# ── Run both pipelines ────────────────────────────────────────────────────────

results = []

print(f"Running on {len(test_data)} examples...")

for i, example in enumerate(test_data):
    utterance   = example["utterance"]
    context     = str(example["context"]).strip()
    instruction = str(example["instruction"]).strip()
    true_label  = pol_classes[example["label_pol"]]

    # ── Oracle metadata (Stage 3) ─────────────────────────────────────────
    oracle_reg, oracle_pow, oracle_tone = get_oracle_metadata(example)

    text_s3 = (f"[register: {oracle_reg}] [power: {oracle_pow}] "
               f"[tone: {oracle_tone}] {utterance} </s> {context} </s> {instruction}")
    pred_s3 = pol_classes[predict_single(model_pol, text_s3)]

    # ── Predicted metadata (Stage 4 original) ────────────────────────────
    # A: register from utterance only
    pred_reg  = reg_classes[predict_single(model_A, utterance)]

    # B: power from utterance + context
    text_B    = utterance + " </s> " + context
    pred_pow  = pow_classes[predict_single(model_B, text_B)]

    # C: tone from utterance + context
    text_C    = utterance + " </s> " + context
    pred_tone = tone_classes[predict_single(model_C, text_C)]

    # Politeness from predicted metadata
    text_s4 = (f"[register: {pred_reg}] [power: {pred_pow}] "
               f"[tone: {pred_tone}] {utterance} </s> {context} </s> {instruction}")
    pred_s4 = pol_classes[predict_single(model_pol, text_s4)]

    results.append({
        "id": i,
        "utterance": utterance,
        "context": context,
        "true_label": true_label,
        # oracle
        "oracle_reg": oracle_reg,
        "oracle_pow": oracle_pow,
        "oracle_tone": oracle_tone,
        "pred_stage3": pred_s3,
        "stage3_correct": pred_s3 == true_label,
        # predicted
        "pred_reg": pred_reg,
        "pred_pow": pred_pow,
        "pred_tone": pred_tone,
        "pred_stage4": pred_s4,
        "stage4_correct": pred_s4 == true_label,
    })

    if (i + 1) % 50 == 0:
        print(f"  {i+1}/330 done...")

print("Done.")

# Save
with open("error_analysis.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print("Saved error_analysis.json")

Running on 330 examples...
  50/330 done...
  100/330 done...
  150/330 done...
  200/330 done...
  250/330 done...
  300/330 done...
Done.
Saved error_analysis.json


In [8]:
# ── Summary ───────────────────────────────────────────────────────────────────

s3_correct = sum(r["stage3_correct"] for r in results)
s4_correct = sum(r["stage4_correct"] for r in results)

s3_preds = [r["pred_stage3"] for r in results]
s4_preds = [r["pred_stage4"] for r in results]
true_labels = [r["true_label"] for r in results]

print(f"Stage 3 accuracy: {s3_correct/330:.4f}  macro F1: {f1_score(true_labels, s3_preds, average='macro'):.4f}")
print(f"Stage 4 accuracy: {s4_correct/330:.4f}  macro F1: {f1_score(true_labels, s4_preds, average='macro'):.4f}")

# ── Key disagreements: Stage 3 correct, Stage 4 wrong ────────────────────────
disagreements = [r for r in results if r["stage3_correct"] and not r["stage4_correct"]]
print(f"\nStage 3 correct but Stage 4 wrong: {len(disagreements)} examples")

# ── Print top 10 most informative disagreements ───────────────────────────────
print("\n" + "="*80)
print("TOP DISAGREEMENTS — Stage 3 correct, Stage 4 wrong")
print("="*80)

for r in disagreements[:10]:
    print(f"\nID {r['id']} | True: {r['true_label']}")
    print(f"Utterance: {r['utterance']}")
    print(f"Context:   {r['context']}")
    print(f"Oracle   → reg:{r['oracle_reg']:12} pow:{r['oracle_pow']:22} tone:{r['oracle_tone']:10} → {r['pred_stage3']}")
    print(f"Predicted → reg:{r['pred_reg']:12} pow:{r['pred_pow']:22} tone:{r['pred_tone']:10} → {r['pred_stage4']}")
    print(f"Metadata errors: ", end="")
    errs = []
    if r['pred_reg']  != r['oracle_reg']:  errs.append(f"register({r['oracle_reg']}→{r['pred_reg']})")
    if r['pred_pow']  != r['oracle_pow']:  errs.append(f"power({r['oracle_pow']}→{r['pred_pow']})")
    if r['pred_tone'] != r['oracle_tone']: errs.append(f"tone({r['oracle_tone']}→{r['pred_tone']})")
    print(", ".join(errs) if errs else "none (politeness model confused despite correct metadata)")
    print("-"*80)

Stage 3 accuracy: 0.8485  macro F1: 0.8007
Stage 4 accuracy: 0.7273  macro F1: 0.7020

Stage 3 correct but Stage 4 wrong: 49 examples

TOP DISAGREEMENTS — Stage 3 correct, Stage 4 wrong

ID 18 | True: neutral
Utterance: ဟင့်အင်း၊ မသိဘူး။
Context:   Non-verbal emphasized speech.
Oracle   → reg:colloquial   pow:equal                  tone:neutral    → neutral
Predicted → reg:slang        pow:equal                  tone:formal     → informal
Metadata errors: register(colloquial→slang), tone(neutral→formal)
--------------------------------------------------------------------------------

ID 34 | True: informal
Utterance: မင်းကတော့ ဆရာ့ဆရာပဲ။
Context:   To an expert.
Oracle   → reg:slang        pow:inferior_to_superior   tone:formal     → informal
Predicted → reg:colloquial   pow:inferior_to_superior   tone:formal     → polite
Metadata errors: register(slang→colloquial)
--------------------------------------------------------------------------------

ID 36 | True: neutral
Utterance: မသိဘူး၊

In [9]:
# ── Which metadata errors cause the most damage? ─────────────────────────────

from collections import Counter

reg_errors  = sum(1 for r in results if r["pred_reg"]  != r["oracle_reg"])
pow_errors  = sum(1 for r in results if r["pred_pow"]  != r["oracle_pow"])
tone_errors = sum(1 for r in results if r["pred_tone"] != r["oracle_tone"])

print(f"Metadata error rates on 330 test examples:")
print(f"  Register errors: {reg_errors}/330  ({reg_errors/330:.1%})")
print(f"  Power errors:    {pow_errors}/330  ({pow_errors/330:.1%})")
print(f"  Tone errors:     {tone_errors}/330  ({tone_errors/330:.1%})")

# How often does each metadata error lead to wrong politeness?
reg_caused  = sum(1 for r in disagreements if r["pred_reg"]  != r["oracle_reg"])
pow_caused  = sum(1 for r in disagreements if r["pred_pow"]  != r["oracle_pow"])
tone_caused = sum(1 for r in disagreements if r["pred_tone"] != r["oracle_tone"])

print(f"\nIn the {len(disagreements)} Stage3✓/Stage4✗ cases:")
print(f"  Register wrong:  {reg_caused}  ({reg_caused/len(disagreements):.1%})")
print(f"  Power wrong:     {pow_caused}  ({pow_caused/len(disagreements):.1%})")
print(f"  Tone wrong:      {tone_caused}  ({tone_caused/len(disagreements):.1%})")

# Most common true label in disagreements
label_counts = Counter(r["true_label"] for r in disagreements)
print(f"\nMost affected true labels:")
for label, count in label_counts.most_common():
    print(f"  {label}: {count} cases")

# Most common Stage 4 mistake per true label
print(f"\nCommon Stage 4 confusions (true → predicted):")
confusions = Counter((r["true_label"], r["pred_stage4"]) for r in disagreements)
for (true, pred), count in confusions.most_common(8):
    print(f"  {true} → {pred}: {count} cases")

Metadata error rates on 330 test examples:
  Register errors: 86/330  (26.1%)
  Power errors:    86/330  (26.1%)
  Tone errors:     109/330  (33.0%)

In the 49 Stage3✓/Stage4✗ cases:
  Register wrong:  31  (63.3%)
  Power wrong:     23  (46.9%)
  Tone wrong:      30  (61.2%)

Most affected true labels:
  neutral: 34 cases
  informal: 10 cases
  polite: 2 cases
  blunt: 2 cases
  rude: 1 cases

Common Stage 4 confusions (true → predicted):
  neutral → polite: 21 cases
  neutral → informal: 7 cases
  informal → polite: 5 cases
  neutral → rude: 5 cases
  informal → neutral: 5 cases
  blunt → neutral: 2 cases
  polite → neutral: 1 cases
  neutral → blunt: 1 cases


In [11]:
import json
from sklearn.metrics import classification_report, f1_score
from collections import Counter

CLASSES = ['blunt', 'informal', 'neutral', 'polite', 'professional', 'rude']

# ── Load results ──────────────────────────────────────────────────────────────
with open("results_ChatGPT_o4_mini.json") as f:
    chatgpt_results = json.load(f)

with open("results_Gemini_3_Fast.json") as f:
    gemini_results = json.load(f)

def analyze_llm(results, model_name):
    true_labels = [r["true_label"] for r in results]
    pred_labels = [r["predicted_label"] for r in results]

    print(f"\n{'='*70}")
    print(f"{model_name} — Per-class Results")
    print('='*70)
    print(classification_report(true_labels, pred_labels,
                                labels=CLASSES, zero_division=0))

    # ── Label distribution of predictions ────────────────────────────────────
    pred_dist = Counter(pred_labels)
    true_dist = Counter(true_labels)
    print("Label distribution (true vs predicted):")
    print(f"{'Label':<15} {'True':>8} {'Predicted':>10} {'Diff':>8}")
    print("-" * 45)
    for label in CLASSES:
        t = true_dist[label]
        p = pred_dist[label]
        print(f"{label:<15} {t:>8} {p:>10} {p-t:>+8}")

    # ── Most common confusions ────────────────────────────────────────────────
    confusions = Counter(
        (r["true_label"], r["predicted_label"])
        for r in results if not r["correct"]
    )
    print(f"\nTop confusions (true → predicted):")
    for (true, pred), count in confusions.most_common(8):
        print(f"  {true:<15} → {pred:<15} : {count} cases")

    # ── Classes never predicted ───────────────────────────────────────────────
    never_predicted = [c for c in CLASSES if pred_dist[c] == 0]
    if never_predicted:
        print(f"\nClasses never predicted: {never_predicted}")

analyze_llm(chatgpt_results, "ChatGPT o4-mini")
analyze_llm(gemini_results,  "Gemini 3 Fast")

# ── Side by side macro F1 per class ──────────────────────────────────────────
print(f"\n{'='*70}")
print("Per-class F1 comparison")
print('='*70)
print(f"{'Class':<15} {'Stage3':>8} {'Stage4':>8} {'ChatGPT':>9} {'Gemini':>8}")
print("-" * 52)

# Stage 3 and Stage 4 per-class F1 from classification reports
stage3_f1 = {'blunt':0.67, 'informal':0.83, 'neutral':0.87,
             'polite':0.82, 'professional':0.94, 'rude':0.67}
stage4_f1 = {'blunt':0.70, 'informal':0.71, 'neutral':0.77,
             'polite':0.67, 'professional':0.91, 'rude':0.46}

chatgpt_true = [r["true_label"] for r in chatgpt_results]
chatgpt_pred = [r["predicted_label"] for r in chatgpt_results]
gemini_true  = [r["true_label"] for r in gemini_results]
gemini_pred  = [r["predicted_label"] for r in gemini_results]

for label in CLASSES:
    c_f1 = f1_score(chatgpt_true, chatgpt_pred, labels=[label],
                    average='macro', zero_division=0)
    g_f1 = f1_score(gemini_true,  gemini_pred,  labels=[label],
                    average='macro', zero_division=0)
    print(f"{label:<15} {stage3_f1[label]:>8.2f} {stage4_f1[label]:>8.2f} "
          f"{c_f1:>9.2f} {g_f1:>8.2f}")


ChatGPT o4-mini — Per-class Results
              precision    recall  f1-score   support

       blunt       0.32      0.86      0.47        14
    informal       0.43      0.95      0.59        59
     neutral       1.00      0.29      0.45       174
      polite       0.55      0.77      0.64        56
professional       0.65      1.00      0.79        15
        rude       0.91      0.83      0.87        12

    accuracy                           0.56       330
   macro avg       0.64      0.78      0.63       330
weighted avg       0.77      0.56      0.54       330

Label distribution (true vs predicted):
Label               True  Predicted     Diff
---------------------------------------------
blunt                 14         37      +23
informal              59        131      +72
neutral              174         50     -124
polite                56         78      +22
professional          15         23       +8
rude                  12         11       -1

Top confusions (tr